In [ ]:
import json
from pathlib import Path
import pymupdf4llm
import re
from bs4 import BeautifulSoup
from html import unescape


BASE_DIR = Path(r"D:\AI Engineer Assignment\Assignment")

PDF_DIR = BASE_DIR / "data/pdfs"

OUTPUT_DIR = BASE_DIR / "extracted"
MARKDOWN_DIR = BASE_DIR / "extracted_md"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)

def clean_html(text: str) -> str:

    # Decode HTML entities
    text = unescape(text)

    # Remove formatting tags but preserve their content
    text = re.sub(
        r"</?(?:u|b|i|strong|em|span|font|sup|sub)[^>]*>",
        "",
        text,
        flags=re.IGNORECASE
    )

    # Convert HTML line-break and paragraph tags into newlines
    text = re.sub(r"<br\s*/?>", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"</p\s*>", "\n", text, flags=re.IGNORECASE)

    # Remove remaining HTML tags
    text = re.sub(r"<[^>]+>", "", text)

    # Clean spaces without destroying Markdown tables
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n[ \t]*\n{2,}", "\n\n", text)

    return text.strip()
def extract_pdf(pdf_path: Path):

    print(f"\n{'=' * 70}")
    print(f"Processing: {pdf_path.name}")
    print(f"{'=' * 70}")

    doc_id = pdf_path.stem


    output_json = OUTPUT_DIR / f"{doc_id}.json"
    output_md = MARKDOWN_DIR / f"{doc_id}.md"

    if output_json.exists():
        print(f"Already extracted. Skipping: {pdf_path.name}")
        return {
            "status": "skipped",
            "doc_id": doc_id
        }

    pages = pymupdf4llm.to_markdown(
        str(pdf_path),
        page_chunks=True,
        write_images=False,
        show_progress=True
    )

    # --------------------------------------------------------
    # Normalize page output
    # --------------------------------------------------------

    extracted_pages = []

    for idx, page in enumerate(pages, start=1):

        meta = page.get("metadata", {})

        page_num = (
            meta.get("page_number")
            or meta.get("page")
            or idx
        )
        raw_text = page.get("text", "")

        cleaned_text = clean_html(raw_text)

        extracted_pages.append({
            "page": page_num,
            "text": cleaned_text
        })


    result = {
        "doc_id": doc_id,
        "source_file": pdf_path.name,
        "page_count": len(extracted_pages),
        "pages": extracted_pages
    }

    with open(output_json,"w",encoding="utf-8") as f:
        json.dump(
            result,
            f,
            indent=2,
            ensure_ascii=False
        )

    # --------------------------------------------------------
    # Save readable Markdown
    # --------------------------------------------------------

    with open(
        output_md,
        "w",
        encoding="utf-8"
    ) as f:

        for item in extracted_pages:

            f.write(
                f"\n\n--- PAGE {item['page']} ---\n\n"
            )

            f.write(item["text"])

    print(
        f"Done: {doc_id} "
        f"({len(extracted_pages)} pages)"
    )

    return {
        "status": "success",
        "doc_id": doc_id,
        "pages": len(extracted_pages)
    }


def main():

    pdf_files = sorted(PDF_DIR.glob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files.")

    results = []
    errors = []

    for i, pdf_path in enumerate(pdf_files, start=1):

        print(
            f"\n[{i}/{len(pdf_files)}] "
            f"{pdf_path.name}"
        )

        try:

            result = extract_pdf(pdf_path)
            results.append(result)

        except Exception as e:

            error = {
                "file": pdf_path.name,
                "error": str(e)
            }

            errors.append(error)

            print(
                f"ERROR processing {pdf_path.name}:"
            )
            print(str(e))

    # --------------------------------------------------------
    # Save error report
    # --------------------------------------------------------

    error_file = BASE_DIR / "extraction_errors.json"

    with open(
        error_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            errors,
            f,
            indent=2,
            ensure_ascii=False
        )

    successful = sum(
        r["status"] == "success"
        for r in results
    )

    skipped = sum(
        r["status"] == "skipped"
        for r in results
    )

    print("\n" + "=" * 70)
    print("EXTRACTION COMPLETE")
    print("=" * 70)

    print(f"Total PDFs : {len(pdf_files)}")
    print(f"Extracted  : {successful}")
    print(f"Skipped    : {skipped}")
    print(f"Failed     : {len(errors)}")

    if errors:
        print(f"\nSee: {error_file}")


if __name__ == "__main__":
    print("startting")
    main()